# Export MongoDB collections to Parquet

Writes `companies` and `financial_data` to Parquet under `data/parquet/`, for
use as one of four data sources in the engine benchmark.

The export reads from MongoDB rather than from `enheter_alle.json`, because
`financial_data` exists only in MongoDB and reading both from one source
guarantees the variants see identical content.

**Both collections are now exported at full width.** The earlier version wrote
fourteen scalar columns and excluded the nested `data` statement blob, which
meant Parquet's column-pruning advantage was banked at export time rather than
measured at query time — a limitation the previous notebook acknowledged. With
the full schema, pruning is exercised where it belongs, inside the query, and
the benchmark can compare narrow and wide workloads on the same files. The only
fields omitted are `links`, `organisasjonsform.links` and
`foretaksformIHjemlandet.links`, all three empty in 1,171,373 of 1,171,373
records. See `schemas.py` for the derivation.

**Writes are staged through container-local disk.** Spark's committer does not
write files where they belong: it writes them under `_temporary` and renames
them into place, and on this mount those renames fail intermittently because
Dropbox holds the files it is uploading. The failure lands after
`mode("overwrite")` has already deleted the previous export, which is how one
interrupted run left no `financial_data` mirror at all. Spark now writes to
`/tmp` inside the container and the finished files are copied onto the mount,
so Dropbox no longer has to be paused. See `staged_write.py`.

In [1]:
import os

from bootstrap import MONGO_DB, PARQUET_DIR, mongo_db, start_spark
from schemas import COMPANIES_SCHEMA, FINANCIAL_SCHEMA
from staged_write import write_staged

# Paths, the session and the JVM readout all come from bootstrap.py, so the
# five notebooks that open a session document the environment identically.
# Driver memory, thread count, the Mongo connector package and the connection
# URI still come from jupyter/spark-defaults.conf, which is baked into the
# image: nothing about the JVM is configured here, so the notebook cannot drift
# from the environment a grader gets. Change the config file and rebuild.
spark = start_spark("group13_parquet_export")

db = mongo_db(spark)

spark_version        4.2.0
spark_master         local[4]
driver_max_heap_gb   8.0 GB
default_parallelism  4
cpu_cores            12
connector            org.mongodb.spark:mongo-spark-connector_2.13:11.1.0
mongo_uri            mongodb://mongodb:27017


## Change detection

A full re-export takes minutes, so it is skipped when the source is unchanged.
When the signature differs the whole collection is rewritten. At this size,
merge logic would add failure modes to save a few minutes.

Note that widening the schema does not change the signature, so the first run
after this change needs `FORCE_REFRESH = True ` to discard the narrow export.

In [2]:
import json
import os

from mirrors import load_metadata, save_metadata, source_signature, stale_collections

METADATA_PATH = os.path.join(PARQUET_DIR, "_export_metadata.json")

# Both collections are mirrored to Parquet. The signature is per collection, so
# an unchanged one is skipped while the other is rewritten.
COLLECTIONS = ["companies", "financial_data"]

# True for the first run after the schema was widened; the signature alone
# cannot detect a schema change.
FORCE_REFRESH = False

previous = load_metadata(METADATA_PATH, COLLECTIONS)
current = source_signature(db, COLLECTIONS)
stale = stale_collections(current, previous["signature"], force=FORCE_REFRESH)

print(json.dumps(current, indent=2))
print()
for name, needed in stale.items():
    print("%-16s %s" % (name, "export needed" if needed else "unchanged, skipping"))

{
  "companies": {
    "count": 1171373
  },
  "financial_data": {
    "count": 1170292,
    "max_fetched_at": "2026-09-07T20:25:20.796000"
  }
}

companies        unchanged, skipping
financial_data   unchanged, skipping


In [3]:
def export(collection, schema):
    df = (
        spark.read.format("mongodb")
        .option("database", MONGO_DB)
        .option("collection", collection)
        .schema(schema)
        .load()
    )
    target = os.path.join(PARQUET_DIR, collection)
    # Spark writes to container-local disk and the finished files are copied
    # onto the mount. Committing directly into data/ fails intermittently:
    # Dropbox holds the files it is uploading, and the committer's renames then
    # fail with an IOException, after mode("overwrite") has already deleted the
    # previous export. See staged_write.py.
    write_staged(df, target, "parquet")
    # Count from the written files rather than the DataFrame, which would
    # otherwise re-read the whole collection from MongoDB a second time.
    return spark.read.parquet(target).count()


os.makedirs(PARQUET_DIR, exist_ok=True)
recorded = dict(previous["rows_written"])
rows = {}
wrote = False

for name, schema in [("companies", COMPANIES_SCHEMA), ("financial_data", FINANCIAL_SCHEMA)]:
    target = os.path.join(PARQUET_DIR, name)
    if stale[name]:
        print("Exporting %s ..." % name)
        rows[name] = export(name, schema)
        wrote = True
        print("  wrote %d rows" % rows[name])
    elif name not in recorded and os.path.exists(target):
        # Unchanged, so nothing to write - but no row count was ever recorded
        # for it either, and a reader cannot tell that from a mirror that does
        # not exist. Counting Parquet reads footers, not data, so this is cheap.
        rows[name] = spark.read.parquet(target).count()
        print("Skipping %s (unchanged); backfilled row count %d" % (name, rows[name]))
    else:
        print("Skipping %s (unchanged)" % name)

# Written on every run, not only on runs that exported something: a mirror
# checked and found current is worth recording as current. save_metadata carries
# forward the entries this run did not touch, so the benchmark's staleness check
# can distinguish "unchanged" from "never exported".
recorded = save_metadata(METADATA_PATH, current, rows, previous, wrote=wrote)
print()
print("Recorded rows per mirrored collection: %s" % recorded)

Skipping companies (unchanged)
Skipping financial_data (unchanged)

Recorded rows per mirrored collection: {'financial_data': 1170292, 'companies': 1171373}


## Verification

Row counts must match MongoDB exactly, and the widened columns must be
populated at the frequencies the profiling pass recorded. A count match alone
would not catch a nested field that read as null throughout.

In [4]:
from pyspark.sql import functions as F

# Nothing below is a stored constant. Every expectation is measured from
# MongoDB in the same run, so the checks keep working as the corpus grows and
# a fetch run cannot invalidate them.

# Field paths to profile on both sides. MongoDB rejects "." in $group output
# names, so each path also gets a flat alias used only as an aggregation key.
FIELDS = [
    ("naeringskode1.kode", "nk1_kode"),
    ("forretningsadresse.kommunenummer", "adr_kommunenummer"),
    ("kapital.belop", "kapital_belop"),
    ("naeringskode3.kode", "nk3_kode"),
]


def mongo_profile(collection, fields):
    """Row count plus per-field non-null counts, in a single server-side pass.

    $ifNull collapses missing and explicit null to the same thing, which is how
    Spark's isNotNull reads the same documents.
    """
    group = {"_id": None, "rows": {"$sum": 1}}
    for path, alias in fields:
        group[alias] = {"$sum": {"$cond": [
            {"$eq": [{"$ifNull": ["$" + path, None]}, None]}, 0, 1]}}
    return db[collection].aggregate([{"$group": group}]).next()


comp = spark.read.parquet(os.path.join(PARQUET_DIR, "companies"))
fin = spark.read.parquet(os.path.join(PARQUET_DIR, "financial_data"))

# One aggregation pass per collection rather than one scan per column.
mongo_comp = mongo_profile("companies", FIELDS)
mongo_fin = mongo_profile("financial_data", [("data", "data")])

spark_comp = comp.select(
    F.count(F.lit(1)).alias("rows"),
    *[F.sum(F.col(path).isNotNull().cast("long")).alias(alias)
      for path, alias in FIELDS]
).collect()[0]

spark_fin = fin.select(
    F.count(F.lit(1)).alias("rows"),
    F.sum(F.col("data").isNotNull().cast("long")).alias("data"),
).collect()[0]

# A row count alone would not catch a nested field that read as null throughout,
# so the widened columns are compared as well.
rows = [("companies.rows", spark_comp["rows"], mongo_comp["rows"]),
        ("financial_data.rows", spark_fin["rows"], mongo_fin["rows"]),
        ("financial_data.data", spark_fin["data"], mongo_fin["data"])]
rows += [("companies." + path, spark_comp[alias], mongo_comp[alias])
         for path, alias in FIELDS]

print("%-40s %12s %12s" % ("metric", "parquet", "mongo"))
print("-" * 68)
failures = []
for label, sp, mg in rows:
    ok = sp == mg
    print("%-40s %12d %12d %s" % (label, sp, mg, "OK" if ok else "MISMATCH"))
    if not ok:
        failures.append(label)

assert not failures, "parquet does not match MongoDB: %s" % ", ".join(failures)

# Reported, not asserted: reaching into element 0 of an array is where Spark's
# and MongoDB's null handling are least obviously equivalent, so a difference
# here is worth reading rather than aborting on.
n = fin.filter(F.col("data")[0]["resultatregnskapResultat"]["totalresultat"]
               .isNotNull()).count()
print("\n%-40s %12d %12s" % ("data[0]...totalresultat", n, "(not compared)"))

size = sum(os.path.getsize(os.path.join(d, fn))
           for d, _, files in os.walk(PARQUET_DIR) for fn in files)
print("\nParquet total on disk: %.2f GB" % (size / 1024**3))

metric                                        parquet        mongo
--------------------------------------------------------------------
companies.rows                                1171373      1171373 OK
financial_data.rows                           1170292      1170292 OK
financial_data.data                            445225       445225 OK
companies.naeringskode1.kode                  1135646      1135646 OK
companies.forretningsadresse.kommunenummer      1108985      1108985 OK
companies.kapital.belop                        437842       437842 OK
companies.naeringskode3.kode                     1576         1576 OK



data[0]...totalresultat                        241962 (not compared)



Parquet total on disk: 0.35 GB
